# Silver template: clean, key and upsert one table (Snowflake)

Reads one Bronze table, cleans it to an explicit contract, gives every row its surrogate key from the
central key table, and upserts it into Silver. Copy this notebook, fill in the **Parameters** cell, and run it.

**Standards this notebook follows**

1. **Layers.** one database per layer: `BRONZE`, `SILVER`, `GOLD`, with keys in `KEYS`. Tables are named `<DATABASE>.<SCHEMA>.<TABLE>`.
2. **Contract.** Only the columns in `columns` are kept, each cast to its type. No `SELECT *`.
3. **Surrogate keys.** `<NAME>_SK` comes from the key table `KEYS.SK.<NAME>`: the same business key gets the
   same key from every source, and a rebuild never renumbers it. The key column's default is a Snowflake
   **sequence** (`<NAME>_SEQ.NEXTVAL`), so the `MERGE` only inserts business keys. Snowflake doesn't enforce
   unique constraints, so the `MERGE` is what keeps business keys unique. The Unknown member is `-1`.
4. **No key, no row.** Rows with a missing business key go to `<TARGET>_QUARANTINE` with a reason.
5. **Clean.** Text is trimmed, blank text becomes NULL, chosen columns are lower-cased, and the newest row
   per business key wins (`QUALIFY ROW_NUMBER()` by the watermark).
6. **Incremental.** Reads rows with `watermark >=` the newest one already in Silver; the MERGE makes
   re-reading them harmless. (A stream on the Bronze table is the alternative when Bronze is append-only.)
7. **Upsert, never overwrite.** `MERGE` on the surrogate key; a row is updated only when its `_ROW_HASH`
   changed and its watermark isn't older than the stored one.
8. **Audit columns** on every row: `_ROW_HASH`, `_RUN_ID`, `_LOADED_AT`.

**Before the first run** a keys admin creates the key table once (this notebook only adds business keys to it):

```sql
CREATE SEQUENCE IF NOT EXISTS KEYS.SK.CUSTOMER_SEQ START = 1 INCREMENT = 1 ORDER;
CREATE TABLE IF NOT EXISTS KEYS.SK.CUSTOMER (
    CUSTOMER_SK NUMBER(10,0) NOT NULL DEFAULT KEYS.SK.CUSTOMER_SEQ.NEXTVAL,
    CUSTOMER_BK VARCHAR      NOT NULL,
    CREATE_DATE TIMESTAMP_NTZ NOT NULL,
    CONSTRAINT PK_CUSTOMER PRIMARY KEY (CUSTOMER_SK),
    CONSTRAINT UQ_CUSTOMER_BK UNIQUE (CUSTOMER_BK)
);
INSERT INTO KEYS.SK.CUSTOMER (CUSTOMER_SK, CUSTOMER_BK, CREATE_DATE) VALUES (-1, 'UNKNOWN', CURRENT_TIMESTAMP());
```

## Who supports this notebook

This is an **enterprise template** (`silver-upsert` 1.0.0, Snowflake). The line between enterprise and domain is drawn by the cells:

| Cells | Marked | Who owns them |
| --- | --- | --- |
| **Parameters** | `# kratosforge:parameters` | The **domain**: table names, keys, columns, measures. Values only, no code. |
| Every other code cell | `# kratosforge:standard` | The **enterprise**: it maintains, fixes and upgrades them. Don't edit them. |
| Markdown cells | none | Anyone: add notes freely. |

While the standard cells are unchanged and no code cell is added, the notebook **conforms** and the enterprise can
take it over: check it with `fabric-desktop templates check <notebook>`, then hand it over with
`fabric-desktop templates handover <notebook> --domain <domain> --owner <you>`.
Editing a standard cell, or adding a code cell, makes it **domain-owned** ad hoc work: it still runs, but the
domain supports it, and the check lists each change. Need logic the template doesn't have? Put it in Silver
(a derived column) or ask the enterprise to add it to the template, so every domain gets it.

In [ ]:
# kratosforge:parameters template=silver-upsert platform=snowflake version=1.0.0 | The domain fills in this cell: values only, no code.
# Parameters
source_table = "BRONZE.CRM.CUSTOMERS"
target_table = "SILVER.CRM.CUSTOMERS"
key_table = "KEYS.SK.CUSTOMER"               # issues CUSTOMER_SK for CUSTOMER_BK
business_key = ["customer_id"]               # the columns that identify a row in the source system
watermark_column = "updated_at"              # when the source row last changed
columns = {                                  # the Silver contract: every column kept, and its type
    "customer_id": "number(10,0)",
    "name": "varchar",
    "segment": "varchar",
    "city": "varchar",
    "email": "varchar",
    "updated_at": "date",
}
lowercase_columns = ["email"]
run_id = ""          # the task or orchestrator passes its run id; one is made when empty
full_reload = False  # True re-reads the whole source (after a logic change or a backfill)

In [ ]:
# kratosforge:standard template=silver-upsert platform=snowflake version=1.0.0 cell=setup | Enterprise-managed: change the Parameters cell, not this one.
import re
import uuid
from datetime import datetime, timezone

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.exceptions import SnowparkSQLException

session = get_active_session()
UNKNOWN_SK = -1  # the Unknown member: rows whose key can't be found point here, never at NULL
_NAME = re.compile(r"[A-Za-z_][A-Za-z0-9_$]{0,254}")
_TYPE = re.compile(r"(?i)(varchar|string|text|boolean|number\(\d{1,2}(,\s*\d{1,2})?\)|int|integer|bigint|float|double|date|timestamp_ntz|timestamp_tz)")
_RUN_ID = re.compile(r"[A-Za-z0-9_.:-]{1,64}")


def q(name: str) -> str:
    """A table name, checked: parameters arrive from tasks, so nothing else can reach the SQL."""
    parts = name.split(".")
    if not 1 <= len(parts) <= 3 or not all(_NAME.fullmatch(x) for x in parts):
        raise ValueError(f"Not a table name: {name!r}")
    return ".".join(parts).upper()


def c(name: str) -> str:
    if not _NAME.fullmatch(name):
        raise ValueError(f"Not a column name: {name!r}")
    return name.upper()


def checked_type(dtype: str) -> str:
    if not _TYPE.fullmatch(dtype.strip()):
        raise ValueError(f"Not a supported type: {dtype!r}")
    return dtype.strip().upper()


def run(sql: str) -> list:
    """Runs one statement. DML and DDL return one summary row; collect() is how Snowpark runs them."""
    return session.sql(sql).collect()


def table_exists(name: str) -> bool:
    try:
        session.table(q(name)).schema  # resolved now, so a missing table raises here
        return True
    except SnowparkSQLException:
        return False


def row_hash(columns: list, alias: str = "") -> str:
    """SHA-256 of the columns, NULL-safe: a change in any of them changes the hash."""
    a = f"{alias}." if alias else ""
    parts = [f"COALESCE(TO_VARCHAR({a}{c(x)}), CHR(0))" for x in columns] or ["''"]
    return f"SHA2(ARRAY_TO_STRING(ARRAY_CONSTRUCT({', '.join(parts)}), CHR(31)), 256)"


def is_text(dtype: str) -> bool:
    return dtype.upper().split("(")[0] in ("VARCHAR", "STRING", "TEXT")


run_id = run_id or str(uuid.uuid4())
if not _RUN_ID.fullmatch(run_id):
    raise ValueError("run_id may only hold letters, digits and _ . : -")
loaded_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
LOADED_AT = f"TO_TIMESTAMP_NTZ('{loaded_at}')"
print(f"Run {run_id} at {loaded_at} UTC")

## 1. Read what changed, cleaned to the contract

In [ ]:
# kratosforge:standard template=silver-upsert platform=snowflake version=1.0.0 cell=step-1 | Enterprise-managed: change the Parameters cell, not this one.
key_name = key_table.rsplit(".", 1)[-1]
sk_column, bk_column = c(f"{key_name}_sk"), c(f"{key_name}_bk")
quarantine_table = f"{target_table}_QUARANTINE"
for name in [*business_key, watermark_column, *lowercase_columns]:
    if name not in columns:
        raise ValueError(f"{name} must be one of the columns in the contract")
columns = {c(n): checked_type(t) for n, t in columns.items()}
business_key = [c(k) for k in business_key]
lowercase_columns = [c(n) for n in lowercase_columns]
wm = c(watermark_column)

audit_ddl = ["_ROW_HASH VARCHAR(64)", "_RUN_ID VARCHAR", "_LOADED_AT TIMESTAMP_NTZ"]
run(f"""CREATE TABLE IF NOT EXISTS {q(target_table)} (
    {", ".join([f"{sk_column} NUMBER(10,0) NOT NULL", *[f"{n} {t}" for n, t in columns.items()], *audit_ddl])},
    CONSTRAINT PK_{c(key_name)} PRIMARY KEY ({sk_column}))""")


def cleaned(name: str, dtype: str) -> str:
    if not is_text(dtype):
        return f"CAST({name} AS {dtype}) AS {name}"
    text = f"NULLIF(TRIM({name}), '')"
    return f"{'LOWER(' + text + ')' if name in lowercase_columns else text} AS {name}"


since = "" if full_reload else f"WHERE {wm} >= (SELECT MAX({wm}) FROM {q(target_table)}) OR NOT EXISTS (SELECT 1 FROM {q(target_table)})"
run(f"""CREATE OR REPLACE TEMPORARY TABLE SILVER_INCOMING AS
SELECT {", ".join(cleaned(n, t) for n, t in columns.items())}
FROM {q(source_table)}
{since}""")

## 2. Quarantine rows with no business key, then keep the newest row per key

In [ ]:
# kratosforge:standard template=silver-upsert platform=snowflake version=1.0.0 cell=step-2 | Enterprise-managed: change the Parameters cell, not this one.
missing_key = " OR ".join(f"{k} IS NULL" for k in business_key)
cols = ", ".join(columns)
run(f"""CREATE TABLE IF NOT EXISTS {q(quarantine_table)} AS
SELECT {cols}, CAST(NULL AS VARCHAR) AS _REASON, CAST(NULL AS VARCHAR) AS _RUN_ID, CAST(NULL AS TIMESTAMP_NTZ) AS _LOADED_AT
FROM SILVER_INCOMING WHERE 1 = 0""")
rejected = run(f"""INSERT INTO {q(quarantine_table)} ({cols}, _REASON, _RUN_ID, _LOADED_AT)
SELECT {cols}, 'business key missing', '{run_id}', {LOADED_AT} FROM SILVER_INCOMING WHERE {missing_key}""")
if rejected and rejected[0][0]:
    print(f"{rejected[0][0]} rows had no business key; they are in {quarantine_table}")

# The newest row per business key wins.
run(f"""CREATE OR REPLACE TEMPORARY TABLE SILVER_STAGED AS
SELECT {cols} FROM SILVER_INCOMING
WHERE NOT ({missing_key})
QUALIFY ROW_NUMBER() OVER (PARTITION BY {", ".join(business_key)} ORDER BY {wm} DESC NULLS LAST) = 1""")

## 3. Surrogate keys from the key table

In [ ]:
# kratosforge:standard template=silver-upsert platform=snowflake version=1.0.0 cell=step-3 | Enterprise-managed: change the Parameters cell, not this one.
if not table_exists(key_table):
    raise RuntimeError(f"{key_table} doesn't exist yet. A keys admin creates it once (see the first cell).")


def bk_sql(alias: str) -> str:
    """The business key as one string: trimmed, blank as NULL. Composite keys join with || and escape |."""
    parts = [f"NULLIF(TRIM(TO_VARCHAR({alias}.{k})), '')" for k in business_key]
    if len(parts) == 1:
        return parts[0]
    parts = [f"REPLACE({x}, '|', CHR(92) || '|')" for x in parts]
    return "CONCAT(" + ", '||', ".join(parts) + ")"


# 1. New business keys are added; the sequence gives them their keys. Existing ones keep theirs.
run(f"""MERGE INTO {q(key_table)} AS k
USING (SELECT DISTINCT {bk_sql('b')} AS BK FROM SILVER_STAGED b) AS s
ON k.{bk_column} = s.BK
WHEN NOT MATCHED AND s.BK IS NOT NULL THEN INSERT ({bk_column}, CREATE_DATE) VALUES (s.BK, CURRENT_TIMESTAMP())""")

# 2. Every row takes its key. A row that still has none would be a bug, so the run stops.
run(f"""CREATE OR REPLACE TEMPORARY TABLE SILVER_KEYED AS
SELECT COALESCE(k.{sk_column}, {UNKNOWN_SK}) AS {sk_column}, {", ".join(f"b.{n}" for n in columns)},
       {row_hash(list(columns), 'b')} AS _ROW_HASH, '{run_id}' AS _RUN_ID, {LOADED_AT} AS _LOADED_AT
FROM SILVER_STAGED b
LEFT JOIN {q(key_table)} k ON k.{bk_column} = {bk_sql('b')}""")
if run(f"SELECT COUNT(*) FROM SILVER_KEYED WHERE {sk_column} = {UNKNOWN_SK}")[0][0]:
    raise RuntimeError(f"Some rows found no key in {key_table}; nothing was written")

## 4. Upsert into Silver

In [ ]:
# kratosforge:standard template=silver-upsert platform=snowflake version=1.0.0 cell=step-4 | Enterprise-managed: change the Parameters cell, not this one.
every = [sk_column, *columns, "_ROW_HASH", "_RUN_ID", "_LOADED_AT"]
result = run(f"""MERGE INTO {q(target_table)} AS t
USING SILVER_KEYED AS s
ON t.{sk_column} = s.{sk_column}
WHEN MATCHED AND t._ROW_HASH <> s._ROW_HASH AND (t.{wm} IS NULL OR s.{wm} >= t.{wm}) THEN
    UPDATE SET {", ".join(f"t.{n} = s.{n}" for n in every if n != sk_column)}
WHEN NOT MATCHED THEN
    INSERT ({", ".join(every)}) VALUES ({", ".join(f"s.{n}" for n in every)})""")
print(f"Silver MERGE: {result[0].as_dict() if result else 'done'}")

## 5. Check the result

In [ ]:
# kratosforge:standard template=silver-upsert platform=snowflake version=1.0.0 cell=step-5 | Enterprise-managed: change the Parameters cell, not this one.
summary = run(f"""SELECT COUNT(*) AS ROW_COUNT, COUNT(DISTINCT {sk_column}) AS KEY_COUNT,
       COUNT_IF({sk_column} = {UNKNOWN_SK}) AS UNKNOWN_COUNT
FROM {q(target_table)}""")[0]
if summary["ROW_COUNT"] != summary["KEY_COUNT"] or summary["UNKNOWN_COUNT"]:
    raise RuntimeError(f"{target_table} has duplicate or Unknown keys: {summary.as_dict()}")
print(f"{target_table}: {summary['ROW_COUNT']} rows, one per {sk_column}")